# Imports

In [ ]:
import pandas as pd
import numpy as np
import gc
import optuna
from optuna.samplers import TPESampler
from datetime import datetime as dt
from datetime import date
from sklearn.metrics import  root_mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import operator
warnings.filterwarnings("ignore")
# принципиально импортировать mlflow после настройки окружающей среды
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")

import hydroeval as he
from lumod import tools
from lumod.models import HBV

/home/cluster/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Earlystopping

In [2]:
class EarlyStoppingCallback(object):
    """Early stopping callback for Optuna."""

    def __init__(self, early_stopping_rounds: int, direction: str = "minimize") -> None:
        self.early_stopping_rounds = early_stopping_rounds

        self._iter = 0

        if direction == "minimize":
            self._operator = operator.lt
            self._score = np.inf
        elif direction == "maximize":
            self._operator = operator.gt
            self._score = -np.inf
        else:
            ValueError(f"invalid direction: {direction}")

    def __call__(self, study: optuna.Study, trial: optuna.Trial) -> None:
        """Do early stopping."""
        if self._operator(study.best_value, self._score):
            self._iter = 0
            self._score = study.best_value
        else:
            self._iter += 1

        if self._iter >= self.early_stopping_rounds:
            study.stop()

In [3]:
def get_hydro_data_on_index(index_hydro, hydro_data):
    hydro_data = hydro_data.loc[hydro_data["index"] == index_hydro]
    hydro_data['precip_amount_org'] = hydro_data['precip_amount_org'].fillna(0)
    hydro_data.set_index("date", inplace=True)
    hydro_data = hydro_data.rename(columns={"q": "qt", "avg_air_temp_org": "tmean", "precip_amount_org": "prec"})
    hydro_data = hydro_data[["prec", "tmean", "qt", "pet"]]
    return hydro_data

In [4]:
def get_post_info(df, index):
    df = df.loc[df["ids"] == index]
    area = float(df['Площ'])
    lat = float(df['lat'])
    return area, lat

In [5]:
hydro_meteo_data = pd.read_excel("data/qtpet_pairs_2.xlsx")
hydro_post_data = pd.read_excel("data/hydrology_stations.xlsx")

In [24]:
index_hydro = 11124

In [25]:
hydro_data = get_hydro_data_on_index(index_hydro, hydro_meteo_data)

In [26]:
hydro_data

,prec,tmean,qt,pet
date,,,,
1995-01-01,0.0,-23.7,4.04,0.0
1995-01-02,0.0,-25.2,3.98,0.0
1995-01-03,0.0,-23.3,3.91,0.0
1995-01-04,0.0,-22.4,3.84,0.0
1995-01-05,0.0,-22.2,3.77,0.0
...,...,...,...,...
2021-12-27,0.0,-17.7,19.80,0.0
2021-12-28,0.0,-19.9,19.60,0.0
2021-12-29,0.0,-26.0,19.50,0.0


#  ML Flow name

In [32]:
mlflow.set_experiment(f"{hydro_post_data.loc[hydro_post_data["ids"] == index_hydro]["name"].values[0]}({index_hydro}) RMSE TPE v4. 2012 по 2021")

2026/01/24 10:56:51 INFO mlflow.tracking.fluent: Experiment with name 'р. Буктырма – с. Берель(11124) RMSE TPE v4. 2012 по 2021' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-artifacts/48', creation_time=1769252211991, experiment_id='48', last_update_time=1769252211991, lifecycle_stage='active', name='р. Буктырма – с. Берель(11124) RMSE TPE v4. 2012 по 2021', tags={}>

# calibration function

In [29]:
def model_objective(trial, area, lat, date_start_minus_one, date_end, hydro_data):
    with mlflow.start_run() as _:
        # Параметры которые необходимо откалибровать для Optuna 
        parameters =  { 
                        # "maxbas": trial.suggest_int("maxbas", 1, 10, step = 1),
                        "tthres": trial.suggest_float("tthres", -2, 5, step = 0.5),
                        "dd": trial.suggest_float("dd", 1, 6, step=0.1),
                        "beta": trial.suggest_float("beta", 1, 6, step=0.1),
                        "fc": trial.suggest_int("fc", 10, 1000, step=10),
                        "k0":  trial.suggest_float("k0", 0.01, 9, step=0.02),
                        "k1": trial.suggest_float("k1", 0.001, 2, step=0.002),
                        "k2": trial.suggest_float("k2", 0.0001, 0.01, step=0.0002),
                        "kp": trial.suggest_float("kp", 0, 2, step = 0.05), # recession coefficient of percolation (1/d)
                        "snow0": trial.suggest_float("snow0", -10, 100, step=2),
                        "w01": trial.suggest_int("w01", 10, 900, step=10),
                        "w02": trial.suggest_int("w02", 10, 500, step=10),
                        "lthres": trial.suggest_int("lthres", 10, 100, step=5),
                        'pwp': trial.suggest_float("pwp", 0.1, 1, step=0.1), # Soil permanente wilting point as a fraction of fc (-) [0,1]  
                        "s0": trial.suggest_float("s0", 0.1, 1, step=0.1), # Initial soil moisture storage (s/fc) [0-1]
                        "cevp": trial.suggest_float("cevp", 0.5, 30, step=0.5),# PET parameter that depends of land use (mm/day.°C)
                        # "cevpam": trial.suggest_float("cevpam", 0, 0.5, step=0.1),
                        # "cevpph": trial.suggest_int("cevpph", 0, 365, step=10),
                }
        # инициализация модели HBV
        model_HBV = HBV(
            area=area,
            lat=lat,
            params=parameters)
        
        # запуск модели
        sim_HBV = model_HBV.run(hydro_data.loc[date_start_minus_one:date_end])
        df_hydro = pd.DataFrame()
        df_hydro["Observed values"] =  hydro_data.loc[date_start_minus_one:date_end].qt
        df_hydro["Simulated values"] =  sim_HBV.loc[date_start_minus_one:date_end].qt
        df_hydro = df_hydro[date_start_minus_one.replace(year=date_start_minus_one.year + 1):]
        df_hydro = df_hydro.dropna()
        df_hydro = df_hydro[df_hydro["Simulated values"] != np.inf]
        df_hydro = df_hydro[df_hydro["Simulated values"] > -np.inf]
        
        if len(df_hydro) > 0:
            mlflow.log_params(parameters)
            metrics = {
                "NSE" : float((he.evaluator(he.nse, df_hydro["Simulated values"].values, df_hydro["Observed values"].values))[0]), 
                "KGE": float((he.evaluator(he.kge, df_hydro["Simulated values"].values, df_hydro["Observed values"].values))[0]),
                "RMSE": root_mean_squared_error(df_hydro["Observed values"].values, df_hydro["Simulated values"].values),
                "MAE": mean_absolute_error(df_hydro["Observed values"].values, df_hydro["Simulated values"].values),
            }
            mlflow.log_metrics(metrics)
            obj_metric = metrics['RMSE']
            del model_HBV
            del sim_HBV
            gc.collect()
            return obj_metric
        return 1000000

In [30]:
start=date(2011, 1, 1)
end=date(2021, 12, 31)
area, lat = get_post_info(hydro_post_data, index_hydro)

In [ ]:
early_stopping = EarlyStoppingCallback(5000, direction="minimize")
sampler_tpe = TPESampler(seed=15, multivariate=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(study_name="study",
                            direction="minimize",
                            sampler=sampler_tpe)

study.optimize(
    lambda trial: model_objective(trial, area=area, lat=lat, date_start_minus_one=start, date_end = end, hydro_data = hydro_data),
    n_trials=10000,
    n_jobs=10,
    gc_after_trial=True,
    show_progress_bar=True,
    callbacks=[early_stopping]
)
best_calibration = study.best_trial.params

In [ ]:
study.best_trial.params

In [16]:
model_HBV = HBV(
    area=area,
    lat=lat,
    params=study.best_trial.params)

# запуск модели
sim_HBV = model_HBV.run(hydro_data.loc[start:end])
df_hydro_calibration = pd.DataFrame()
df_hydro_calibration["Observed values"] =  hydro_data.loc[start:end].qt
df_hydro_calibration["Simulated values"] =  sim_HBV.loc[start:end].qt
df_hydro_calibration = df_hydro_calibration[start.replace(year=start.year + 1):]
df_hydro_calibration
df_hydro_calibration = df_hydro_calibration.dropna()
df_hydro_calibration = df_hydro_calibration[df_hydro_calibration["Simulated values"] != np.inf]
df_hydro_calibration = df_hydro_calibration[df_hydro_calibration["Simulated values"] > -np.inf]

# Visualisation

In [17]:
df_hydro_calibration.to_excel(f"result/{hydro_post_data.loc[hydro_post_data["ids"] == index_hydro]["name"].values[0]}({index_hydro}) RMSE TPE v1. 2012 по 2021.xlsx")

In [ ]:
metrics_calibration = {
        "NSE" : round(float((he.evaluator(he.nse, df_hydro_calibration["Simulated values"].values, df_hydro_calibration["Observed values"].values))[0]), 2), 
        "KGE": round(float((he.evaluator(he.kge, df_hydro_calibration["Simulated values"].values, df_hydro_calibration["Observed values"].values))[0]), 2),
        "RMSE": round(root_mean_squared_error(df_hydro_calibration["Observed values"].values, df_hydro_calibration["Simulated values"].values), 2),
        "MAE": round(mean_absolute_error(df_hydro_calibration["Observed values"].values, df_hydro_calibration["Simulated values"].values), 2),
    }
metrics_calibration = pd.DataFrame.from_dict([metrics_calibration])
metrics_calibration

In [ ]:
numeric_cols = ['Observed values', 'Simulated values']
years = sorted(df_hydro_calibration.index.year.unique())

# Оптимальное количество колонок
ncols = 2
nrows = (len(years) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols, 
    figsize=(20, nrows*2.5), 
    constrained_layout=True
)

# fig.suptitle('Калибровка ЭКОМАГ', fontsize=16, y=1.02)
date_format = mdates.DateFormatter("%b")

# Заглушки для легенды
lines = []
for col in numeric_cols:
    line, = axes.flat[0].plot([], [], label=col)
    lines.append(line)

# Отрисовка графиков
for i, year in enumerate(years):
    ax = axes.flat[i]
    year_data = df_hydro_calibration[df_hydro_calibration.index.year == year]

    ax.plot(year_data['Observed values'], color="#1f77b4", linewidth=1.3)
    ax.plot(year_data['Simulated values'], color="#d62728", linewidth=1.3)
    ax.set_ylabel("Q, m³/s", fontsize=9)
    ax.set_title(f'{year} year', fontsize=9)
    ax.xaxis.set_major_formatter(date_format)
    ax.grid(True, linestyle='--', alpha=0.4)

# Убираем пустые оси
for j in range(i+1, nrows*ncols):
    fig.delaxes(axes.flat[j])

# Легенду переносим вниз
# fig.legend(
#     handles=lines,
#     labels=numeric_cols,
#     loc='upper left',
#     ncol=2,
#     frameon=False,
#     fontsize=12
# )
# Добавляем отступ снизу под легенду
plt.subplots_adjust(bottom=0.10)

# plt.savefig("annual_comparison.pdf", dpi=300, bbox_inches="tight")
plt.show()